# 04 — Why Green Evaluators Can Lie
## Detector Adequacy, Epistemic Collapses, and Fail-Closed Verification Auditing

**Series:** `001_finite_obstruction_calculus` / `012_lggr_i_evaluated_path_kernels`  
**Author:** Jeremy H. Carroll  
**Status:** Living Educational Supplement  
**Anchor Tag:** `sealed-derivation-v1` (`3100c4e`, cut 2026-08-17)

---

> ### **Claim-Typing Legend**
> - **[Pinned]**: Verified directly by executable Python functions in `code/epk_core.py` and passing test suites.
> - **[Proved]**: Mechanized in Lean 4 under `PaperAE004` (reported kernel axioms: `propext`, `Classical.choice`, `Quot.sound` or axiom-free).
> - **[Declared]**: An operational definition, architectural parameter, or illustrative teaching model chosen by the author.
> - **[Open]**: A mathematical or empirical question not discharged in this package.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

nb_dir = Path.cwd().resolve()
if (nb_dir / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir))
elif (nb_dir / "jupyter_notebooks" / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir / "jupyter_notebooks"))
from epk_nb import import_epk, widgets_available

epk = import_epk()
HAS_WIDGETS = widgets_available()

from epk_core import *  # shipped names; notebooks must not reimplement the engine

print("Python ready.")
print("epk_core imported from shipped code/ (filesystem path omitted).")
print("HAS_WIDGETS =", HAS_WIDGETS)


In [ ]:
def explain_delta(e1, e2, eg):
    d = calculate_evaluator_defect(int(e1), int(e2), int(eg))
    kind = {
        -1: "NGD — local steps green, composed task red",
        0: "ZDR — local product agrees with global",
        1: "PGD — local product red, global green",
    }[d]
    print(f"E1={int(e1)}  E2={int(e2)}  Eg={int(eg)}  =>  Delta_E={d}  ({kind})")
    return d

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    print("Interactive: flip local and global verdicts. Headless runs skip this widget.")
    widgets.interact(explain_delta, e1=[0, 1], e2=[0, 1], eg=[0, 1])
else:
    print("Headless fallback — every Boolean triple:")
    for e1 in (0, 1):
        for e2 in (0, 1):
            for eg in (0, 1):
                explain_delta(e1, e2, eg)


## 1. The Core Warning: Factorization $\neq$ Correctness

A central temptation in automated verification is to assume that if a pipeline shows **Zero Evaluator Defect** ($\Delta_E \equiv 0$) or **Zero Kernel Error** ($\mathrm{RKE} = 0$), the system must be completely sound.

This assumption is dangerously wrong.

Recall the manuscript's central finding:
> **Local green is not global correctness. Evaluator factorization is the condition under which declared local verdicts compose into the declared global verdict; factorization alone is not evidence of semantic correctness.**

### Why can a broken system factorize?
Consider a lazy or adversarial test engineer who writes evaluators that **always return 1 (Pass)**:
- $E_1(\gamma_1) = 1$ identically.
- $E_2(\gamma_2) = 1$ identically.
- $E_g(\gamma_2 \star \gamma_1) = 1$ identically.

Does it factorize?
$$E_g = 1 = 1 \times 1 = E_1 \cdot E_2 \implies \Delta_E = 0 \quad (\text{identically!})$$

The residual $R_{\mathrm{Eval}}$ is exactly zero! The composition theorem holds perfectly! Yet the agents could be generating completely unparseable garbage, and the test suite would happily stamp it **GREEN**.


## 2. The Eight Epistemic Collapse Modes (§8, §9)

In Sections 8 and 9, the paper catalogs how evaluators and boundary cuts can collapse into degenerate states. To prevent these collapses, `epk_core.py` provides the **`DetectorAdequacyGuard`**:

| Collapse Mode | Condition | Why It Falsifies the Evaluation |
| :--- | :--- | :--- |
| **`VACUOUS_LOCAL`** | $E_1 \equiv 1$ and $E_2 \equiv 1$ on all traces | Local checks never fail; every integration bug is mechanically blamed on evaluator gluing defect ($\Delta_E = -1$). |
| **`ALL_REJECTING`** | $E_1 \equiv 0$ or $E_2 \equiv 0$ on all traces | Local product $E_1 E_2 \equiv 0$. If $E_g \equiv 0$, records $\Delta_E \equiv 0$, deceptively reporting zero defect on a dead system. |
| **`OBSERVED_AGREEMENT`** | $E_g = E_1 E_2$ across all observed traces | Observed sample agreement between local product and global verdict; flags potential tautological coupling or bypassed integration, though sample agreement alone does not prove the implementation code was tautologically copied. |
| **`CONSTANT_GLOBAL_REJECT`** | $E_g \equiv 0$ on all traces | Global test rejects everything unconditionally. |
| **`CONSTANT_GLOBAL_ACCEPT`** | $E_g \equiv 1$ on all traces | Global test accepts everything unconditionally. |
| **`CONSTANT_LOCAL_PRODUCT_ZERO`** | $E_1 E_2 \equiv 0$ on all traces | No trace ever passes both local checks simultaneously. |
| **`TRIVIAL_SLICE`** | Boundary slice $|B| \le 1$ with terminal collapse | Intermediate checkpoint is not a genuine interior state cut (identity or terminal collapse). |
| **`SINGLETON_SLICE_CUT_UNASSESSED`** | Boundary cut $|B| \le 1$ with unconfirmed collapse | A singleton boundary cut without trajectory evidence confirming interior traversal. |


In [ ]:
# 1. Fault Injection: Vacuous Local Validators
# Adversary provides evaluators where local stages pass every trace unconditionally
vacuous_traces = [
    EvaluatorVerdictTriple(e1=1, e2=1, eg=1),
    EvaluatorVerdictTriple(e1=1, e2=1, eg=0),  # NGD
    EvaluatorVerdictTriple(e1=1, e2=1, eg=0),  # NGD
]

flags_vacuous = DetectorAdequacyGuard.audit(vacuous_traces)
print("=== FAULT INJECTION 1: VACUOUS LOCAL VALIDATORS ===")
print("Injected traces where E1=1 and E2=1 identically.")
print(f"Guard Audit Output: {[f.value for f in flags_vacuous]}")
assert DetectorAdequacyStatus.VACUOUS_LOCAL in flags_vacuous
print("[INTERCEPTED] Guard successfully flagged vacuous evaluation!")

# 2. Fault Injection: All-Rejecting Collapse
broken_traces = [
    EvaluatorVerdictTriple(e1=0, e2=1, eg=0),
    EvaluatorVerdictTriple(e1=0, e2=0, eg=0),
]
flags_broken = DetectorAdequacyGuard.audit(broken_traces)
print("\n=== FAULT INJECTION 2: ALL-REJECTING PIPELINE ===")
print(f"Guard Audit Output: {[f.value for f in flags_broken]}")
assert DetectorAdequacyStatus.ALL_REJECTING in flags_broken
print("[INTERCEPTED] Guard flagged false green on dead pipeline!")

# 3. Healthy Baseline: The Canonical Five-Trace Phone Test
healthy_traces = make_five_trace_phone_test()
flags_healthy = DetectorAdequacyGuard.audit(healthy_traces)
print("\n=== HEALTHY EVALUATION: FIVE-TRACE PHONE TEST ===")
print(f"Guard Audit Output: {[f.value for f in flags_healthy]}")
assert flags_healthy == [DetectorAdequacyStatus.NO_LISTED_DEGENERACY_OBSERVED]
print("[CLEARED] No listed degeneracy observed on fixture (detector adequacy is not certified)!")


In [ ]:
# 4. Testing Singleton Boundary Slices
a, b, c = State("Sa"), State("Sb"), State("Sc")
b_slice = BoundarySlice({b})

# Case A: Singleton cut with no trajectory proof -> UNASSESSED
flags_unassessed = DetectorAdequacyGuard.audit(healthy_traces, boundary_slice=b_slice, composed_trajectories=None)
print("=== SINGLETON BOUNDARY CUT: NO TRAJECTORIES ===")
print(f"Guard Audit Output: {[f.value for f in flags_unassessed]}")
assert DetectorAdequacyStatus.SINGLETON_SLICE_CUT_UNASSESSED in flags_unassessed

# Case B: Singleton cut with genuine interior traversal -> UNASSESSED (interior confirmed, not trivial)
comp_interior = ComposedTrajectory(
    TrajectorySegment(a, b, ("step1",), raw_action=1.0),
    TrajectorySegment(b, c, ("step2",), raw_action=1.0)
)
flags_interior = DetectorAdequacyGuard.audit(healthy_traces, boundary_slice=b_slice, composed_trajectories=[comp_interior])
print("\n=== SINGLETON BOUNDARY CUT: GENUINE INTERIOR TRAVERSAL ===")
print(f"Guard Audit Output: {[f.value for f in flags_interior]}")
assert DetectorAdequacyStatus.SINGLETON_SLICE_CUT_UNASSESSED in flags_interior
assert DetectorAdequacyStatus.TRIVIAL_SLICE not in flags_interior

# Case C: Confirmed terminal collapse -> TRIVIAL_SLICE
comp_terminal = ComposedTrajectory(
    TrajectorySegment(a, c, ("step1",), raw_action=1.0),
    TrajectorySegment(c, c, (), raw_action=0.0)
)
flags_terminal = DetectorAdequacyGuard.audit(healthy_traces, boundary_slice=BoundarySlice({c}), composed_trajectories=[comp_terminal])
print("\n=== SINGLETON BOUNDARY CUT: CONFIRMED TERMINAL COLLAPSE ===")
print(f"Guard Audit Output: {[f.value for f in flags_terminal]}")
assert DetectorAdequacyStatus.TRIVIAL_SLICE in flags_terminal
print("\n[VERIFIED] DetectorAdequacyGuard enforces precise singleton distinction!")


In [ ]:
# Compute Evaluation Test Suite Health dynamically from actual audit results
b_slice_terminal = BoundarySlice({State("Sc")})
comp_terminal_case = ComposedTrajectory(
    TrajectorySegment(State("Sa"), State("Sc"), ("step1",), raw_action=1.0),
    TrajectorySegment(State("Sc"), State("Sc"), (), raw_action=0.0)
)

audit_cases_defs = [
    ("No Listed Degeneracy (Phone Test)", make_five_trace_phone_test(), None, None),
    ("Vacuous Local Checks", vacuous_traces, None, None),
    ("All-Rejecting Evaluators", broken_traces, None, None),
    ("Observed Agreement", [EvaluatorVerdictTriple(1, 1, 1), EvaluatorVerdictTriple(0, 1, 0)], None, None),
    ("Constant Global Accept", [EvaluatorVerdictTriple(1, 0, 1), EvaluatorVerdictTriple(0, 1, 1)], None, None),
    ("Terminal Slice Collapse", healthy_traces, b_slice_terminal, [comp_terminal_case]),
]

flag_labels = [
    "NO_LISTED_DEGENERACY_OBSERVED",
    "VACUOUS_LOCAL",
    "ALL_REJECTING",
    "OBSERVED_AGREEMENT_SAMPLE_LEVEL",
    "CONSTANT_GLOBAL_ACCEPT",
    "TRIVIAL_SLICE",
]

audit_cases = [case[0] for case in audit_cases_defs]
status_matrix = np.zeros((len(audit_cases_defs), len(flag_labels)), dtype=int)

for i, (name, traces, b_sl, c_trajs) in enumerate(audit_cases_defs):
    flags = DetectorAdequacyGuard.audit(traces, boundary_slice=b_sl, composed_trajectories=c_trajs)
    for j, flag_name in enumerate(flag_labels):
        target_status = getattr(DetectorAdequacyStatus, flag_name)
        if target_status in flags:
            status_matrix[i, j] = 1

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=120)
im = ax.imshow(status_matrix, cmap="Blues", aspect="auto")

flag_display_labels = [
    "NO_LISTED_DEGENERACY",
    "VACUOUS_LOCAL",
    "ALL_REJECTING",
    "OBSERVED_AGREEMENT",
    "CONSTANT_GLOBAL_ACCEPT",
    "TRIVIAL_SLICE",
]

ax.set_xticks(np.arange(len(flag_display_labels)))
ax.set_yticks(np.arange(len(audit_cases)))
ax.set_xticklabels(flag_display_labels, rotation=35, ha="right", fontsize=9, fontweight="bold")
ax.set_yticklabels(audit_cases, fontsize=9.5, fontweight="bold")
ax.set_title("DetectorAdequacyGuard: Computed Audit & Interception Matrix", fontsize=11, fontweight="bold")

for i in range(len(audit_cases)):
    for j in range(len(flag_labels)):
        text = "FLAGGED" if status_matrix[i, j] == 1 and j > 0 else ("CLEAR" if status_matrix[i, j] == 1 else "")
        color = "white" if status_matrix[i, j] == 1 else "black"
        ax.text(j, i, text, ha="center", va="center", color=color, fontweight="bold", fontsize=8.5)

plt.tight_layout()
plt.show()


## 3. Mandatory Claim Hygiene: CAN-Show vs CANNOT-Show

### What this guard CAN show:
- Detects whether an evaluation suite exhibits known structural collapses (vacuity, all-rejection, observed agreement / potential tautological coupling, constant outputs, or unassessed cuts).
- Note: A clean audit (`NO_LISTED_DEGENERACY_OBSERVED`) confirms the absence of listed structural collapses on the sample, but adequacy is not certified: a green reading is never self-certifying.
- Ensures that zero-defect readings ($R_{\mathrm{Eval}} = 0$) are not celebrated when the evaluation setup is degenerate.
- Provides an audit preflight reporting structural warnings before computing higher-level reliability metrics; downstream pipelines must enforce gating policies when flags are present.

### What this guard CANNOT show:
- Does **not** prove that passing evaluators are semantically optimal or bug-free (a subtle semantic bug can evade both local and global checks without triggering structural collapse).
- Does **not** eliminate the need for human domain review of test cases and prompts.


## 4. Student Exercises

### Exercise 1: Spot the Flaw
An engineering team reports:
*"Our local tests pass 100% of the time ($\mathrm{LPR}=1.0$), and our global test passes 80% of the time ($\mathrm{GPR}=0.80$). Therefore, our agents have an 80% success rate and our evaluation is completely healthy."*

What flag will `DetectorAdequacyGuard` raise, and why is their conclusion flawed?

<details>
<summary><b>Click to see the solution for Exercise 1</b></summary>

- **Flag:** `DetectorAdequacyStatus.VACUOUS_LOCAL`.
- **Flaw:** No local rejection was observed on this sample ($\mathrm{LPR}=1.0$). The sample does not establish that the local evaluators discriminate relevant failure cases. While this does not prove the evaluator code is constant across all possible inputs, it warrants flagging `VACUOUS_LOCAL` and requiring counterexample/rejection evidence before relying on the local verdicts.
- As a consequence, every single one of the $20\%$ global failures is mechanically branded as an evaluator gluing defect ($\Delta_E = -1$, NGD).
- In reality, the agents may simply have written broken code that failed to run. The local tests were vacuous.
</details>

### Exercise 2: Tautological Integration
A developer creates an integration test $E_g$ by calling `assert test_stage1() and test_stage2()`.
What happens to $\Delta_E$, and what does the guard report?

<details>
<summary><b>Click to see the solution for Exercise 2</b></summary>

- Since $E_g := E_1 \cdot E_2$, by algebraic definition:  
  $$\Delta_E = E_g - E_1 E_2 = E_1 E_2 - E_1 E_2 = 0$$
- The gluing defect $\Delta_E$ is identically zero on every possible trace!
- The guard flags `OBSERVED_AGREEMENT_SAMPLE_LEVEL` (or potential tautological coupling), catching the fact that no genuine end-to-end integration distinction was observed on the sample.
</details>


---
## Anchors & Provenance

- **Paper Sections:** §8 (GTOC/GTLA Compliance Note — Detector Adequacy), §9 (Threats to Validity — Threats 6 and 7), §9.5 (Falsification Criteria).
- **Claim Ledger IDs:** H1 (Detector adequacy audit), H2 (Collapse conditions).
- **Code Artifacts:** `epk_core.DetectorAdequacyStatus`, `epk_core.DetectorAdequacyGuard`.
- **Lean 4 Mechanization:**
  - `PaperAE004.H2_structural_guard_implies_ngd_strictly_below_global_failure` in `lean4/lggr_i_evaluated_path_kernels.lean`
  - `PaperAE004.E16_bool_dashboard_cannot_be_exact` in `lean4/lggr_i_evaluated_path_kernels.lean`
- **Next Step:** Open [`05_cancellation_and_poe_vs_moe.ipynb`](05_cancellation_and_poe_vs_moe.ipynb) to investigate aggregate defect cancellation, Product-of-Experts vs Mixture-of-Experts, and verification scaling laws.
